# Gov Mode — popular o catálogo FIPE (marca / modelo / ano)

Este notebook roda a **mesma engine de extração que o painel "Gov Mode" usa**
(`src/govmode/engine.py::GovEngine`, `src/govmode/executor.py`) — não é uma
reimplementação da lógica de busca na API, é o motor de verdade sendo
chamado diretamente, fora do servidor FastAPI. Os dados gravados aqui
aparecem imediatamente nos KPIs, no catálogo e na lista de pipelines da
página `/gov` do app.

**Pré-requisitos** (mesmos do resto do projeto, ver `docs/SPEC.md` seção 13):
- `.env` na raiz com `GOOGLE_APPLICATION_CREDENTIALS` apontando para a
  service account do BigQuery.
- `FIPE_API_TOKEN` no `.env` — necessário só para o passo de código FIPE
  (marcas/modelos/anos funcionam sem ele).
- Rode este notebook com o kernel do `venv` do projeto (mesmo interpretador
  que roda `main.py`).

**Sobre o tempo de execução**: a API da FIPE exige uma pausa entre
chamadas (`PAUSA_ENTRE_CHAMADAS = 0.5s`, para não ser bloqueada por IP) e o
catálogo aqui grava **todos os anos de cada modelo**, não só o mais
recente — para uma marca com muitos modelos isso pode levar vários
minutos. É seguro interromper a execução (parar a célula) e rodar de novo:
a extração é resumível — retoma de onde parou, sem duplicar linhas nem
refazer chamadas já feitas (seção 13.5 do SPEC).


## 1. Setup — caminho do projeto, configuração e status do ambiente

In [1]:
import sys
from pathlib import Path

# Ajuste só se este notebook for movido para fora da raiz do projeto.
RAIZ_PROJETO = Path.cwd()
if not (RAIZ_PROJETO / "src").is_dir():
    RAIZ_PROJETO = RAIZ_PROJETO.parent

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

print("Raiz do projeto:", RAIZ_PROJETO)


Raiz do projeto: c:\Users\guilh\Documents\Project\Git\Ford_Fiap


In [2]:
# `src.config` carrega o .env automaticamente na importação (carregar_env()).
from src.config import settings

print("BigQuery configurado:      ", settings.bigquery_configurado)
print("Dataset do Gov Mode:       ", settings.govmode_bigquery_dataset)
print("FIPE_API_TOKEN configurado:", bool(settings.fipe_api_token))

if not settings.bigquery_configurado:
    print(
        "\n⚠ GOOGLE_APPLICATION_CREDENTIALS não está configurado ou não aponta "
        "para um arquivo válido — as gravações no BigQuery vão falhar. Confira o .env."
    )
if not settings.fipe_api_token:
    print(
        "\n⚠ FIPE_API_TOKEN ausente — os passos de marcas/modelos/meses de "
        "referência funcionam normalmente, mas o passo de código FIPE vai falhar "
        "com 'MissingToken'. Adicione o token no .env para popular modelos_com_fipe."
    )


BigQuery configurado:       True
Dataset do Gov Mode:        analytics_ford
FIPE_API_TOKEN configurado: True


Cria as 4 tabelas do catálogo no BigQuery caso ainda não existam — sem
isso, uma consulta (seções 6/7) ou uma execução parcial (ex.: sem
`FIPE_API_TOKEN`, o passo de código FIPE nunca chega a gravar e a tabela
`modelos_com_fipe` nunca é criada) pode esbarrar em `NotFound: Table ...
was not found`. `garantir_tabelas()` é idempotente — rodar de novo não tem
efeito colateral.


In [3]:
from src.govmode import warehouse

if settings.bigquery_configurado:
    warehouse.garantir_tabelas()
    print("Tabelas do catálogo prontas (criadas agora ou já existentes).")


Tabelas do catálogo prontas (criadas agora ou já existentes).


## 2. Parâmetros — edite aqui antes de rodar

`MARCAS_ALVO` é obrigatória e não pode ficar vazia (mesma regra do painel:
rodar "todas as marcas do Brasil" de uma vez multiplicaria demais o volume
de chamadas do passo de código FIPE). Os nomes precisam bater com os
nomes que a própria API da FIPE usa — não importa maiúsculas/minúsculas
(ex.: `"ford"` casa com `"Ford"`).


In [4]:
TIPO_VEICULO = "carros"          # carros | motos | caminhoes
MARCAS_ALVO = ["Ford"]           # nomes de marca, ex.: ["Ford", "Chevrolet"]

NOME_PIPELINE = f"Extração FIPE — {', '.join(MARCAS_ALVO)}"


## 3. Criar (ou reaproveitar) a pipeline no painel

Isso registra uma pipeline nomeada de verdade em `data/gov_pipelines.json`
— a mesma estrutura que o botão "Nova pipeline" da página `/gov` cria. Ela
passa a aparecer na lista de "Pipelines de extração" do painel, com
histórico de execuções e código de DAG geráveis por lá também.

Rodar esta célula de novo é seguro: se já existir uma pipeline com esse
nome, ela é **atualizada** (só gera uma versão nova se marca-alvo/tipo de
veículo realmente mudou) em vez de duplicada.


In [5]:
from src.govmode import store
from src.govmode.models import (
    ConfigSchedule,
    FonteGovFipe,
    GovPipelineEntrada,
    PassoGovPipeline,
    STEPS_PADRAO_GOV,
)

fonte = FonteGovFipe(tipo_veiculo=TIPO_VEICULO, marcas_alvo=MARCAS_ALVO)
steps = [PassoGovPipeline(type=t) for t in STEPS_PADRAO_GOV]

existente = next((p for p in store.listar_pipelines() if p.name == NOME_PIPELINE), None)

entrada = GovPipelineEntrada(
    name=NOME_PIPELINE,
    description="Criada pelo notebook gov_mode_popular_fipe.ipynb",
    source=fonte,
    steps=steps,
    schedule=ConfigSchedule(mode="manual"),
)

if existente:
    pipeline = store.atualizar_pipeline(existente.id, entrada)
    print(f"Pipeline existente reaproveitada: {pipeline.id} (v{pipeline.version})")
else:
    pipeline = store.criar_pipeline(entrada)
    print(f"Pipeline criada: {pipeline.id} (v{pipeline.version})")


Pipeline existente reaproveitada: gov_pipeline_90f4eeb8f351 (v1)


## 4. Executar a extração

Chama `executor.executar_agora(...)` — a mesma função que o botão
"Executar" da página `/gov` dispara, só que aguardada de forma síncrona
aqui em vez de rodar em background. Ao final, o resultado já fica
persistido em `data/gov_runs.json` e a pipeline mostra a última execução
no painel.

`await` funciona direto na célula porque o Jupyter roda sobre seu próprio
loop assíncrono (ipykernel ≥ 6) — não precisa de `asyncio.run()`.


In [6]:
from src.govmode.executor import executar_agora

run = await executar_agora(
    pipeline.snapshot(),
    pipeline_id=pipeline.id,
    pipeline_name=pipeline.name,
    pipeline_version=pipeline.version,
    origem="notebook",
)

print("Status geral:", run.status.value)
print()
for etapa in run.steps:
    linha = f"[{etapa.status.value:9}] {etapa.name}"
    if etapa.resumo:
        linha += f" — {etapa.resumo}"
    if etapa.error:
        linha += f" — ERRO ({etapa.error.error_type}): {etapa.error.message}"
    print(linha)

if run.progresso:
    print()
    print("Progresso detalhado:", run.progresso)


Status geral: partial

[success  ] Marcas — 107 marca(s) de 'carros'
[success  ] Modelos — 0 modelo(s) novo(s) de 1 marca(s)
[success  ] Código FIPE — 122 código(s) FIPE novo(s) — 497/497 modelo(s) processado(s), 298 chamada(s) com falha
[failed   ] Meses de referência — ERRO (FipeAPIError): API respondeu 500 em 'https://brasilapi.com.br/api/fipe/tabelas/v1': {"message":"Fonte de dados FIPE temporariamente indisponível. Tente novamente mais tarde.","type":"internal_error","name":"Error"}

Progresso detalhado: {'fipe': {'feito': 497, 'total': 497}}


## 5. (Opcional) Rodar de novo para avançar mais

Como a extração é resumível, basta **rodar a célula da seção 4 de novo**
quantas vezes quiser — cada execução retoma de onde a anterior parou
(modelos e anos já resolvidos não são buscados de novo). Útil se a
primeira rodada foi interrompida ou se o passo de código FIPE ainda não
processou todos os modelos.

Para adicionar mais marcas depois, edite `MARCAS_ALVO` na seção 2, rode a
célula da seção 3 de novo (atualiza a pipeline existente) e depois a da
seção 4.


## 6. Conferir o que foi gravado no BigQuery

In [7]:

from src.govmode import schemas, warehouse

for tabela in (
    schemas.MARCAS_VEICULOS,
    schemas.MODELOS_VEICULOS,
    schemas.MODELOS_COM_FIPE,
    schemas.REFERENCIA_MESES_FIPE,
):
    try:
        total = warehouse.query(f"SELECT COUNT(*) AS total FROM `{warehouse.table_id(tabela)}`")
        print(f"{tabela:22} {int(total.iloc[0]['total'])} linha(s)")
    except Exception as exc:
        print(f"{tabela:22} ainda sem dado ({type(exc).__name__})")


marcas_veiculos        107 linha(s)
modelos_veiculos       497 linha(s)
modelos_com_fipe       1022 linha(s)
referencia_meses_fipe  0 linha(s)


In [8]:
# Amostra do catálogo já com código FIPE resolvido, para a(s) marca(s) desta execução.
from google.cloud import bigquery

amostra = warehouse.query(f"""
    SELECT marca_codigo, modelo_nome, ano_modelo, codigo_fipe
    FROM `{warehouse.table_id(schemas.MODELOS_COM_FIPE)}`
    WHERE tipo_veiculo = @tipo
    ORDER BY atualizado_em DESC
    LIMIT 20
""", [bigquery.ScalarQueryParameter("tipo", "STRING", TIPO_VEICULO)])

amostra


,marca_codigo,modelo_nome,ano_modelo,codigo_fipe
0,22,Fiesta Supercharger 1.0 8V 95cv 5p,2003 Gasolina,003260-3
1,22,Fiesta Supercharger 1.0 8V 95cv 5p,2004 Gasolina,003260-3
2,22,Fiesta Street/ Action 1.0 8v 5p,2005 Gasolina,003244-1
3,22,Fiesta TIT./TIT.Plus 1.6 16V Flex Aut.,2018 Flex,003390-1
4,22,Fiesta Street 1.0 8v 3p,2001 Gasolina,003229-8
5,22,Fiesta Street 1.6 8v 95cv 5p,2002 Gasolina,003256-5
6,22,Fiesta Supercharger 1.0 8V 95cv 5p,2005 Gasolina,003260-3
7,22,Fiesta Street 1.6 8v 95cv 5p,2003 Gasolina,003256-5
8,22,Fiesta Street/ Action 1.0 8v 5p,2006 Gasolina,003244-1
9,22,Fiesta Street/ Action 1.0 8v 5p,2004 Gasolina,003244-1


## Próximos passos

- Abra `/gov` no app (Flask + FastAPI rodando) para ver os KPIs, o
  catálogo navegável e esta pipeline na lista, com o histórico desta
  execução.
- Pela própria página é possível gerar a DAG do Airflow para agendar essa
  mesma pipeline (botão "Gerar DAG"), sem precisar deste notebook de novo.
